# DP2 Early-Release Injection Smoke Test

This notebook is a lightweight RSP validation run for the injection pipeline.

It is meant to answer one simple question quickly: **can we load a Butler coadd,
inject a small synthetic cluster catalog, and recover a reasonable subset with the
built-in detector?**

Suggested use:

1. Select the `Python (inject-rsp)` kernel tied to your `INJECT` venv.
2. Edit the Butler repo / collection / `DATA_ID` values in the next cell for the
   DP2 early-release dataset you want to test.
3. Run top to bottom.

Important: this notebook requires Rubin Butler on RSP. If you see
`ModuleNotFoundError: No module named 'lsst.daf'`, switch to the
`Python (inject-rsp)` kernel or move to an RSP environment.

The notebook keeps the run intentionally small so it is easy to iterate while you
are still validating access and basic pipeline behavior.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    from lsst.daf.butler import Butler
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Rubin Butler is not available in this notebook kernel. "
        "Use this notebook on RSP with the 'Python (inject-rsp)' kernel, "
        "or use one of the non-Butler notebooks for local runs."
    ) from exc

import inject

required = ['ClusterConfig', 'InjectionConfig', 'InjectionPipeline', 'notebook_output_dir']
missing = [name for name in required if not hasattr(inject, name)]
package_file = getattr(inject, '__file__', None)

if package_file is None or missing:
    raise ImportError(
        "Python imported the wrong 'inject' package for this notebook. "
        f"Imported from {package_file!r} with missing symbols: {missing}. "
        "Reinstall from your INJECT checkout with 'bash scripts/setup_rsp_env.sh', "
        "then restart the notebook kernel."
    )

from inject import ClusterConfig, InjectionConfig, InjectionPipeline, notebook_output_dir
from inject.detection import run_cluster_detection

RUN_OUTPUT_DIR = notebook_output_dir('dp2_early_release_smoke_test')
PLOT_OUTPUT_DIR = RUN_OUTPUT_DIR / 'plots'
print(f'Run outputs: {RUN_OUTPUT_DIR}')
print(f'Using inject from: {package_file}')

## 1. Configure The Dataset And Test Run

Replace the Butler settings below with the DP2 early-release values available in your
RSP environment. The defaults are placeholders based on the older DP0.2 examples in
this repo, so you should expect to edit them for the actual early-release test.

In [ ]:
# ------------------------------------------------------------------
# Update these three lines for the DP2 early-release dataset you want
# to test on RSP.
# ------------------------------------------------------------------
BUTLER_REPO = 'dp02'
BUTLER_COLLECTIONS = '2.2i/runs/DP0.2'
DATA_ID = {'tract': 3828, 'patch': 24, 'band': 'i'}

# Keep this test small and fast while validating the workflow.
N_CLUSTERS = 25
CUTOUT_SIZE = 800
SEED = 42

config = InjectionConfig(
    run_name='dp2_early_release_smoke_test',
    band=DATA_ID['band'],
    cutout_size=CUTOUT_SIZE,
    n_clusters=N_CLUSTERS,
    seed=SEED,
    edge_buffer=50,
    use_actual_psf=True,
    psf_fwhm_fallback=3.5,
    record_psf_mask_flags=True,
    skip_bad_psf_regions=False,
    save_injected_image=False,
    output_dir=str(RUN_OUTPUT_DIR),
    tract=DATA_ID.get('tract'),
    patch=DATA_ID.get('patch'),
    cluster_config=ClusterConfig(
        profile_type='king',
        method='smooth',
        mag_min=20.5,
        mag_max=24.5,
        r_half_min=2.0,
        r_half_max=8.0,
        concentration_min=5.0,
        concentration_max=20.0,
    ),
)

config

## 2. Load A Butler Coadd Cutout

This uses the same packaged `InjectionPipeline.load_data(...)` Butler path as the
main RSP workflows, so it is a good first check that the dataset access pattern is
compatible with this repository.

In [ ]:
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
pipe = InjectionPipeline(config)
pipe.load_data(butler=butler, data_id=DATA_ID)

print(f'Loaded image shape: {pipe.image.shape}')
print(f'BBox origin: {pipe.bboxes[config.band]}')
print(f'PSF object type: {type(pipe.psf_objs[config.band]).__name__}')

vmin, vmax = np.percentile(pipe.image, [1, 99])
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(pipe.image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title(f"Original {config.band}-band cutout")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

## 3. Generate A Small Injection Catalog And Inject It

This is the actual smoke test. If this cell succeeds and the next figure shows compact
positive residuals at the injected positions, the basic pipeline path is working.

In [ ]:
catalog = pipe.generate_catalog()
injected_image, injection_info = pipe.inject(catalog=catalog, drop_stamps=True, verbose=True)

print(f'Catalog size: {len(catalog)}')
print(f'Injected entries kept: {len(injection_info)}')

diff = injected_image.astype(float) - pipe.image.astype(float)
dv = max(np.nanpercentile(np.abs(diff), 99.5), 1e-6)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
axes[0].imshow(pipe.image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('Original')
axes[1].imshow(injected_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[1].set_title('Injected')
axes[2].imshow(diff, cmap='RdBu_r', origin='lower', vmin=-dv, vmax=dv)
axes[2].set_title('Difference')
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

for entry in injection_info[:10]:
    axes[1].plot(entry['x'], entry['y'], 'o', ms=5, mec='lime', mfc='none', mew=0.9)

plt.show()

## 4. Optional Built-In Detection Pass

This uses the repository's built-in matched-filter + MCI detector as a quick validation
tool. It is not meant to be your final science detector; it just gives a fast sanity
check that recoveries are happening at plausible rates.

In [ ]:
psf_samples = [row.get('psf_fwhm_px') for row in injection_info if row.get('psf_fwhm_px') is not None]
detector_psf_fwhm = float(np.nanmedian(psf_samples)) if psf_samples else config.psf_fwhm_fallback

detections = pipe.detect_with(
    run_cluster_detection,
    psf_fwhm=detector_psf_fwhm,
    threshold_sigma=3.0,
    mci_max=0.85,
    snr_min=3.0,
    r_half_min=1.0,
    ellipticity_max=0.6,
    box_size=64,
    pixel_scale=config.pixel_scale,
    use_multiscale=True,
    use_mci=True,
    verbose=False,
)

stats = pipe.analyze(match_radius=5.0)
stats

## 5. Save Outputs And Quick-Look Plots

This writes the injection catalog, detection catalog, and a few standard figures to the
run output directory so you can compare tests across different early-release collections
or data IDs.

In [ ]:
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pipe.save_results()

plot_results = pipe.make_plots(
    output_dir=str(PLOT_OUTPUT_DIR),
    plots=['injection_summary', 'before_after', 'position_map', 'completeness_1d', 'psf_fwhm_hist'],
    show=True,
    save=True,
    n_stamps=min(6, len(pipe.injection_info)),
)

print('Saved files:')
for key, path in plot_results['saved'].items():
    print(f' - {key}: {path}')

print(f'Run directory: {RUN_OUTPUT_DIR}')

## What Success Looks Like

A healthy smoke-test run usually means:

- the Butler load succeeds for your DP2 early-release repo/collection
- the injection cell completes without PSF or mask-plane errors
- the difference image shows compact positive residuals at injected locations
- the built-in detector finds at least some of the brighter / larger injections
- result files appear under the printed run directory

If the load fails, the first thing to check is the Butler repo / collection / `DATA_ID`
combination. If injection fails after loading succeeds, that usually points to a dataset-
specific PSF or mask behavior worth debugging next.